# CNN for sequence learning

In the previous MLP, each nucleotide position was processed independently before pooling — so the model learned nucleotide *composition* but not *order*. A CNN fixes this by sliding a small filter window across the sequence, detecting local *patterns* (motifs) at every position.

Steps:
```text
              RNA sequence
                 "AUGC"
                   │
                   ▼
            One-hot encoding
          (seq_len × 4 matrix)

      A → [1 0 0 0]
      U → [0 1 0 0]
      G → [0 0 1 0]
      C → [0 0 0 1]

                   │
                   ▼
         Transpose → (4 × seq_len)
         (Conv1d expects channels first)

                   │
                   ▼
         ┌─────────────────────┐
         │   Conv1d layer      │
         │  4 → 16 filters     │
         │  kernel size = 3    │
         │  slides across seq  │
         └─────────────────────┘
                   │
                   ▼
               ReLU
                   │
                   ▼
         Global max pooling
         (pick strongest activation
          across all positions)

              shape: (16,)

                   │
                   ▼
         ┌─────────────────────┐
         │   Linear layer      │
         │      16 → 1         │
         └─────────────────────┘
                   │
                   ▼
            Scalar prediction

             expression score
                 e.g. 0.73
```

## Dimensions

`nn.Conv1d` expects input shape `(batch, channels, length)`. For a single sequence of length 7:

```text
one-hot:          (7, 4)
transpose+batch:  (1, 4, 7)
Conv1d(4→16,k=3): (1, 16, 5)
ReLU:             (1, 16, 5)
global max pool:  (1, 16)
Linear(16→1):     (1, 1)
```

The kernel can start at positions 1 through 5, covering every 3-nucleotide window:

```text
pos:  1 2 3 4 5 6 7
seq:  A A A U G C C
      [─────]               window 1
        [─────]             window 2
          [─────]           window 3
            [─────]         window 4
              [─────]       window 5
```

## Global max pooling

After convolution, each of the 16 filters has one activation per window position. Global max pooling collapses the length dimension by keeping the strongest activation across all positions for each filter. This answers: "did this motif appear anywhere in the sequence?" - regardless of where.

- Filter 1 active → motif-like pattern #1 detected
- Filter 2 active → motif-like pattern #2 detected
- …
- Filter 16 active → motif-like pattern #16 detected

## MLP vs CNN

| | MLP | CNN |
|---|---|---|
| Processes positions | independently | in local windows (kernel) |
| Captures | nucleotide composition | local motifs (e.g. AUG start codon) |
| Pooling | mean pooling | global max pooling |


In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random

In [6]:
#  -------- toy example with multiple sequences and targets
sequences = [
    "AAAUGCC",
    "AUGCGAA",
    "UUUGGCA"
]
targets = [
    0.8,
    0.3,
    0.6
]

#  -------- prepare input
VOCAB = "AUCG"
def one_hot_encode(seq: str) -> torch.Tensor:
    seq = seq.upper().strip()
    idx = torch.tensor([VOCAB.index(ch) for ch in seq], 
                       dtype=torch.long)
    mat_eye = torch.eye(len(VOCAB), 
                     dtype=torch.float32)[idx]
    return(mat_eye)

X = [one_hot_encode(seq) for seq in sequences]
Y = torch.tensor(targets, dtype=torch.float32)

# --- minimal CNN model ---
class SeqCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=4, 
                               out_channels=16, 
                               kernel_size=3 #each filter looks at a window of 3 positions
                               )
        self.fc = nn.Linear(16, 1)
    def forward(self, x):
        x = self.conv1(x) # 16 learned features per position
        x = torch.relu(x) # This introduces non-linearity.
        x = torch.max(x, dim=2).values # global max pooling across sequence length
        x = self.fc(x) # scalar prediction
        return x

In [12]:
seq = "AAAUGCC"
x = one_hot_encode(seq)   # (7, 4)
x = x.T.unsqueeze(0)      # (1, 4, 7)
# x.T changes (7, 4) to (4, 7)
# unsqueeze(0) adds batch dimension: (1, 4, 7)
print(f"x={x}")
model = SeqCNN()
pred = model.forward(x)
print(f"pred={pred.item():.2f}")

x=tensor([[[1., 1., 1., 0., 0., 0., 0.],
         [0., 0., 0., 1., 0., 0., 0.],
         [0., 0., 0., 0., 0., 1., 1.],
         [0., 0., 0., 0., 1., 0., 0.]]])
pred=0.03


In [15]:
# And now the whole training loop with batched sequences:
sequences = ["AAAUGCC", "AUGCGAA", "UUUGGCA"]
targets   = [0.8, 0.3, 0.6]

# Define input tensors for all sequences:
X = [one_hot_encode(seq).T.unsqueeze(0) for seq in sequences]
Y = torch.tensor(targets, dtype=torch.float32)
# Or even better, 
# stack all sequences into one batch tensor: (3, 4, 7)
X = torch.stack([one_hot_encode(seq).T for seq in sequences])
Y = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)  # (3, 1)

#  -------- define model
model     = SeqCNN()
loss_fn   = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# -------- training loop --------
# X is now a proper batch, so no inner loop needed:
# model(X) processes all 3 sequences in one forward pass
for epoch in range(10):
    optimizer.zero_grad()
    predictions = model(X)                    # (3, 1)
    loss = loss_fn(predictions, Y) # Measure how wrong the prediction is
    loss.backward() # Compute gradients for all weights.
    optimizer.step() # Optimizer updates the weights using those gradients.
    print(f"epoch={epoch:2d}, loss={loss.item():.2f}, preds={[round(p, 2) for p in predictions.squeeze().tolist()]}")

epoch= 0, loss=0.25, preds=[0.07, 0.12, 0.16]
epoch= 1, loss=0.24, preds=[0.08, 0.13, 0.18]
epoch= 2, loss=0.23, preds=[0.1, 0.14, 0.19]
epoch= 3, loss=0.22, preds=[0.11, 0.16, 0.2]
epoch= 4, loss=0.21, preds=[0.12, 0.17, 0.22]
epoch= 5, loss=0.20, preds=[0.14, 0.19, 0.23]
epoch= 6, loss=0.19, preds=[0.15, 0.2, 0.24]
epoch= 7, loss=0.18, preds=[0.16, 0.21, 0.26]
epoch= 8, loss=0.17, preds=[0.17, 0.23, 0.27]
epoch= 9, loss=0.16, preds=[0.18, 0.24, 0.28]


In [17]:
# Let's predict on a new sequence:
def predict_sequence(seq, model):
    x = one_hot_encode(seq).T.unsqueeze(0)
    model.eval()
    with torch.no_grad():
        pred = model(x)
    return pred.item()

new_seq = "AAAUGCC"
score = predict_sequence(new_seq, model)
print(f"Score: {score:.2f}")

Score: 0.20


## Regularization

### Dropout
Randomly zeros out neurons during training, forcing the network not to rely too heavily on any single one.

```
before:  [0.3  0.7  0.5  0.2]
after:   [0.3  0.0  0.5  0.0]
```

During inference you want deterministic output, so model.eval() automatically disables dropout.

### Batch Normalization
Normalizes activations within each mini-batch (using the batch's own mean and variance), which stabilizes and speeds up training.

During inference there is no batch to compute statistics from, so BatchNorm switches to running averages accumulated during training. model.eval() triggers this switch automatically.

### Summary

In a CNN, batch normalization is commonly added after the convolution and before the activation, while dropout is often added after pooling or before the final dense layer to reduce overfitting.

In [33]:
class SeqCNN(nn.Module):
    def __init__(self, 
                 use_batchnorm=False, 
                 dropout_p=0.0):
        super().__init__()
        self.conv1    = nn.Conv1d(4, 16, kernel_size=3)
        self.bn1      = nn.BatchNorm1d(16) if use_batchnorm else None
        self.dropout  = nn.Dropout(p=dropout_p) if dropout_p > 0 else None
        self.fc       = nn.Linear(16, 1)

    def forward(self, x):
        x = self.conv1(x)
        if self.bn1:     
            x = self.bn1(x)
        x = torch.relu(x)
        x = torch.max(x, dim=2).values
        if self.dropout: 
            x = self.dropout(x)
        return self.fc(x)
    
# We can use nn.Sequential to simplify model definition when 
# layers are in a simple sequence without branching:
class SeqCNN(nn.Module):
    def __init__(self, use_batchnorm=False, dropout_p=0.0):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(4, 16, kernel_size=3),
            nn.BatchNorm1d(16) if use_batchnorm else nn.Identity(),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),   # output: (batch, 16, 1)
            nn.Flatten(),              # output: (batch, 16)
            nn.Dropout(dropout_p) if dropout_p > 0 else nn.Identity(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)

In [34]:
def train_model(X, Y, lr=0.001, n_epochs=10, use_batchnorm=False, dropout_p=0.0):
    model     = SeqCNN(use_batchnorm=use_batchnorm, dropout_p=dropout_p)
    loss_fn   = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, Y)
        loss.backward()
        optimizer.step()
        print(f"epoch={epoch:2d}, loss={loss.item():.2f}")
    return model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("------------ baseline")
m_plain    = train_model(X, Y)

print("------------ + BatchNorm")
m_bn       = train_model(X, Y, use_batchnorm=True)

print("------------ + Dropout")
m_dropout  = train_model(X, Y, dropout_p=0.3)

------------ baseline
epoch= 0, loss=0.49
epoch= 1, loss=0.47
epoch= 2, loss=0.46
epoch= 3, loss=0.44
epoch= 4, loss=0.43
epoch= 5, loss=0.42
epoch= 6, loss=0.40
epoch= 7, loss=0.39
epoch= 8, loss=0.38
epoch= 9, loss=0.36
------------ + BatchNorm
epoch= 0, loss=0.18
epoch= 1, loss=0.17
epoch= 2, loss=0.15
epoch= 3, loss=0.14
epoch= 4, loss=0.13
epoch= 5, loss=0.12
epoch= 6, loss=0.11
epoch= 7, loss=0.10
epoch= 8, loss=0.10
epoch= 9, loss=0.09
------------ + Dropout
epoch= 0, loss=0.30
epoch= 1, loss=0.26
epoch= 2, loss=0.13
epoch= 3, loss=0.11
epoch= 4, loss=0.12
epoch= 5, loss=0.17
epoch= 6, loss=0.18
epoch= 7, loss=0.27
epoch= 8, loss=0.20
epoch= 9, loss=0.15


BatchNorm is usually helpful with real batches. Our training loop uses one sequence at a time, so batch size is effectively 1. Let's try to change that and use `Dataloader`.

In [31]:
from torch.utils.data import Dataset, DataLoader, TensorDataset
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

sequences = ["AAAUGCC", "AUGCGAA", "UUUGGCA",
             "AAAUGCA", "AUGCCCA", "UCCGGCA"]
targets   = [0.8, 0.3, 0.6,
             0.1, 0.2, 0.7]

# Since all sequences have the same length, we can precompute tensors once.
X = torch.stack([one_hot_encode(seq).T for seq in sequences])   # (N, 4, L)
Y = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)     # (N, 1)

dataset = TensorDataset(X, Y)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# ---- Training using DataLoader ----
model = SeqCNN()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    model.train()
    total_loss = 0.0
    for x_batch, y_batch in loader:
        optimizer.zero_grad()
        prediction = model(x_batch)
        loss = loss_fn(prediction, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    print(f"epoch={epoch}, avg_loss={avg_loss:.3f}")

epoch=0, avg_loss=0.472
epoch=1, avg_loss=0.434
epoch=2, avg_loss=0.395
epoch=3, avg_loss=0.364
epoch=4, avg_loss=0.326
epoch=5, avg_loss=0.298
epoch=6, avg_loss=0.272
epoch=7, avg_loss=0.247
epoch=8, avg_loss=0.227
epoch=9, avg_loss=0.202


Alternative: custom RNADataset. If we want encoding to happen inside the dataset instead of precomputing `X` and `Y` first, you can define a custom `Dataset` class and pass it to `DataLoader`.

In [32]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

class RNADataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = sequences
        self.targets = targets

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        y = self.targets[idx]
        x = one_hot_encode(seq).T   # (4, seq_len)
        return x, torch.tensor([y], dtype=torch.float32)

dataset = RNADataset(sequences, targets)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

model = SeqCNN()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    model.train()
    total_loss = 0.0
    for x_batch, y_batch in loader:
        optimizer.zero_grad()
        prediction = model(x_batch)
        loss = loss_fn(prediction, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    print(f"epoch={epoch}, avg_loss={avg_loss:.3f}")

epoch=0, avg_loss=0.472
epoch=1, avg_loss=0.434
epoch=2, avg_loss=0.395
epoch=3, avg_loss=0.364
epoch=4, avg_loss=0.326
epoch=5, avg_loss=0.298
epoch=6, avg_loss=0.272
epoch=7, avg_loss=0.247
epoch=8, avg_loss=0.227
epoch=9, avg_loss=0.202


Both models use **MSE loss** here, since the target is a continuous scalar (regression task). What if it was a *classification* task? Then we should use **BCEWithLogitsLoss**. Then we can apply sigmoid yourselves to convert logits into probabilities.

In [35]:
sequences = [
    "AUGCUAACGU",
    "CCCCGAUUUA",
    "GGGAUACGUA",
    "UUUAAACCCG",
    "ACGUACGUAC",
    "GGGGUUUUAA"
]
labels = [1, 0, 1, 0, 1, 0]
dataset = RNADataset(sequences, labels)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

model = SeqCNN()
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    model.train()
    total_loss = 0.0
    for x_batch, y_batch in loader:
        optimizer.zero_grad()
        prediction = model(x_batch)
        loss = loss_fn(prediction, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    print(f"epoch={epoch}, avg_loss={avg_loss:.3f}")

epoch=0, avg_loss=0.675
epoch=1, avg_loss=0.672
epoch=2, avg_loss=0.669
epoch=3, avg_loss=0.667
epoch=4, avg_loss=0.665
epoch=5, avg_loss=0.663
epoch=6, avg_loss=0.661
epoch=7, avg_loss=0.658
epoch=8, avg_loss=0.656
epoch=9, avg_loss=0.654


In [ ]:
# And prediction, binary output:
model.eval()
with torch.no_grad():
    test_seq = one_hot_encode("AUGGCUACGU").T.unsqueeze(0)  # (1, 4, L)
    logits = model(test_seq)
    probs = torch.sigmoid(logits)
    pred = (probs >= 0.5).int()
    print(f"Probability: {probs.item():.2f}")
    print("Prediction:", pred.item())

Probability: 0.52
Prediction: 1


## Loss functions by task:

- Regression: nn.MSELoss
- Binary classification: nn.BCEWithLogitsLoss
- Multi-class classification: nn.CrossEntropyLoss